In [0]:
CREATE OR REPLACE TABLE workspace.instacart.orders
AS SELECT * FROM read_files(
  '/Volumes/workspace/default/instacart_data/orders.csv',
  format => 'csv',
  header => true,
  inferSchema => true
);

CREATE OR REPLACE TABLE workspace.instacart.products
AS SELECT * FROM read_files(
  '/Volumes/workspace/default/instacart_data/products.csv',
  format => 'csv', header => true, inferSchema => true
);

CREATE OR REPLACE TABLE workspace.instacart.aisles
AS SELECT * FROM read_files(
  '/Volumes/workspace/default/instacart_data/aisles.csv',
  format => 'csv', header => true, inferSchema => true
);

CREATE OR REPLACE TABLE workspace.instacart.departments
AS SELECT * FROM read_files(
  '/Volumes/workspace/default/instacart_data/departments.csv',
  format => 'csv', header => true, inferSchema => true
);

CREATE OR REPLACE TABLE workspace.instacart.order_products_prior
AS SELECT * FROM read_files(
  '/Volumes/workspace/default/instacart_data/order_products__prior.csv',
  format => 'csv', header => true, inferSchema => true
);

CREATE OR REPLACE TABLE workspace.instacart.order_products_train
AS SELECT * FROM read_files(
  '/Volumes/workspace/default/instacart_data/order_products__train.csv',
  format => 'csv', header => true, inferSchema => true
);

CREATE OR REPLACE TABLE workspace.instacart.order_products
AS SELECT * FROM workspace.instacart.order_products_prior
UNION ALL
SELECT * FROM workspace.instacart.order_products_train;

num_affected_rows,num_inserted_rows


In [0]:
SELECT 'orders' AS tbl, COUNT(*) AS rows FROM workspace.instacart.orders
UNION ALL
SELECT 'order_products', COUNT(*) FROM workspace.instacart.order_products
UNION ALL
SELECT 'products', COUNT(*) FROM workspace.instacart.products
UNION ALL
SELECT 'aisles', COUNT(*) FROM workspace.instacart.aisles
UNION ALL
SELECT 'departments', COUNT(*) FROM workspace.instacart.departments;

tbl,rows
orders,3421083
order_products,33819106
products,49688
aisles,134
departments,21


In [0]:
-- Check for invalid values in products table
SELECT *
FROM workspace.instacart.products
WHERE TRY_CAST(aisle_id AS INT) IS NULL
   OR TRY_CAST(department_id AS INT) IS NULL;

product_id,product_name,aisle_id,department_id,_rescued_data
6816,"""Scotch Kids 5"""" Scissors",Blunted,"Red""",null


In [0]:
-- Create cleaned products table
CREATE OR REPLACE TABLE workspace.instacart.products_clean AS
SELECT *
FROM workspace.instacart.products
WHERE TRY_CAST(aisle_id AS INT) IS NOT NULL
  AND TRY_CAST(department_id AS INT) IS NOT NULL;

num_affected_rows,num_inserted_rows


In [0]:
USE workspace.instacart;

In [0]:
--A1 How many total orders exist in the dataset?

select count(*) as total_orders 
from workspace.instacart.orders;

total_orders
3421083


In [0]:
--A2 How many unique customers placed at least one order?
select count(distinct user_id) as unique_customers from workspace.instacart.orders;

unique_customers
206209


In [0]:
--A3 — Total Unique Products
SELECT COUNT(*) as total_products from workspace.instacart.products_clean;

total_products
49687


In [0]:
--A4 — Orders by Day of Week

SELECT order_dow,COUNT(*) as total_orders
FROM workspace.instacart.orders
GROUP BY order_dow
ORDER BY total_orders DESC;

order_dow,total_orders
0,600905
1,587478
2,467260
5,453368
6,448761
3,436972
4,426339


In [0]:
--A5 — Peak Ordering Hours
SELECT order_hour_of_day,COUNT(*) as total_orders
from workspace.instacart.orders
GROUP BY order_hour_of_day
ORDER BY total_orders DESC LIMIT 5;


order_hour_of_day,total_orders
10,288418
11,284728
15,283639
14,283042
13,277999


In [0]:
-- A6: How many products belong to each department?
SELECT d.department,
       COUNT(p.product_id) AS product_count
FROM workspace.instacart.products_clean p
JOIN workspace.instacart.departments d
  ON p.department_id  = d.department_id
GROUP BY d.department
ORDER BY product_count DESC;

department,product_count
personal care,6563
snacks,6264
pantry,5371
beverages,4365
frozen,4007
dairy eggs,3449
household,3084
canned goods,2092
dry goods pasta,1858
produce,1684


In [0]:
--A7 Which 10 products appear most across all orders?
SELECT p.product_name,count(o.order_id)as total_orders
from workspace.instacart.order_products o 
join workspace.instacart.products_clean p
on o.product_id=p.product_id
group by p.product_name
order by total_orders desc limit 10;

product_name,total_orders
Banana,491291
Bag of Organic Bananas,394930
Organic Strawberries,275577
Organic Baby Spinach,251705
Organic Hass Avocado,220877
Organic Avocado,184224
Large Lemon,160792
Strawberries,149445
Limes,146660
Organic Whole Milk,142813


In [0]:
--A8 How many orders contain only 1 product?
SELECT COUNT(*) AS single_item_orders
FROM (
  SELECT order_id
  FROM workspace.instacart.order_products
  GROUP BY order_id
  HAVING COUNT(*) = 1
) t;


single_item_orders
163593


In [0]:
--A9 What is the average basket size (items per order)?
SELECT ROUND(AVG(items_per_order), 2) AS avg_items_per_order
FROM (
  SELECT order_id,
         COUNT(*) AS items_per_order
  FROM workspace.instacart.order_products
  GROUP BY order_id
) t;


avg_items_per_order
10.11


In [0]:
-- A10: Top 5 aisles by product count
SELECT a.aisle,
       COUNT(*) AS product_count
FROM workspace.instacart.products_clean p
JOIN workspace.instacart.aisles a
ON p.aisle_id  = a.aisle_id
GROUP BY a.aisle
ORDER BY product_count DESC
LIMIT 5;

aisle,product_count
missing,1258
candy chocolate,1246
ice cream ice,1091
vitamins supplements,1038
yogurt,1026


In [0]:
--How many total order lines belong to each department?
SELECT d.department,
       COUNT(*) AS total_order_lines
FROM workspace.instacart.order_products op
JOIN workspace.instacart.products_clean p
  ON op.product_id = p.product_id
JOIN workspace.instacart.departments d
  ON p.department_id = d.department_id
GROUP BY d.department
ORDER BY total_order_lines DESC;


department,total_order_lines
produce,9888378
dairy eggs,5631067
snacks,3006412
beverages,2804175
frozen,2336858
pantry,1956819
bakery,1225181
canned goods,1114857
deli,1095540
dry goods pasta,905340


In [0]:
--Reorder Rate by Department
SELECT d.department,
       COUNT(*) AS total_orders,
       SUM(op.reordered) AS reorders,
       ROUND(100.0 * SUM(op.reordered) / COUNT(*), 2)
           AS reorder_rate_pct
FROM workspace.instacart.order_products op
JOIN workspace.instacart.products_clean p ON op.product_id = p.product_id
JOIN workspace.instacart.departments d ON p.department_id = d.department_id
GROUP BY d.department
ORDER BY reorder_rate_pct DESC;


department,total_orders,reorders,reorder_rate_pct
dairy eggs,5631067,3773723,67.02
beverages,2804175,1832952,65.37
produce,9888378,6432596,65.05
bakery,1225181,769880,62.84
deli,1095540,666231,60.81
pets,102221,61594,60.26
babies,438743,253453,57.77
bulk,35932,20736,57.71
snacks,3006412,1727075,57.45
alcohol,159294,90992,57.12


In [0]:
--Overall Reorder Percentage
SELECT COUNT(*) AS total_items,
       SUM(CASE WHEN reordered = 1 THEN 1 ELSE 0 END)
           AS reordered_items,
       ROUND(
           100.0 * SUM(CASE WHEN reordered = 1 THEN 1 ELSE 0 END)
           / COUNT(*), 2
       ) AS reorder_pct
FROM workspace.instacart.order_products;


total_items,reordered_items,reorder_pct
33819106,19955360,59.01


In [0]:
--Customers with 10+ Orders
SELECT COUNT(*) AS loyal_customers
FROM (
  SELECT user_id,
         COUNT(DISTINCT order_id) AS total_orders
  FROM workspace.instacart.orders
  GROUP BY user_id
  HAVING total_orders > 10
) t;


loyal_customers
101696


In [0]:
--Customer Order Segmentation
WITH user_orders AS (
  SELECT user_id,
         COUNT(DISTINCT order_id) AS total_orders
  FROM workspace.instacart.orders
  GROUP BY user_id
)
SELECT
  CASE
    WHEN total_orders = 1 THEN 'One-time'
    WHEN total_orders BETWEEN 2 AND 5 THEN 'Occasional'
    WHEN total_orders BETWEEN 6 AND 15 THEN 'Regular'
    ELSE 'Loyal'
  END AS segment,
  COUNT(*) AS user_count
FROM user_orders
GROUP BY segment
ORDER BY user_count DESC;


segment,user_count
Regular,92744
Loyal,69889
Occasional,43576


In [0]:
--Avg Days Between Orders per Segment
WITH user_stats AS (
  SELECT user_id,
         COUNT(DISTINCT order_id) AS order_count,
         AVG(days_since_prior_order) AS avg_gap
  FROM workspace.instacart.orders
  WHERE days_since_prior_order IS NOT NULL
  GROUP BY user_id
)
SELECT
  CASE
    WHEN order_count = 1 THEN 'One-time'
    WHEN order_count BETWEEN 2 AND 5 THEN 'Occasional'
    WHEN order_count BETWEEN 6 AND 15 THEN 'Regular'
    ELSE 'Loyal'
  END AS segment,
  ROUND(AVG(avg_gap), 1) AS avg_days_between_orders
FROM user_stats
GROUP BY segment
ORDER BY avg_days_between_orders;


segment,avg_days_between_orders
Loyal,10.0
Regular,16.7
Occasional,19.7


In [0]:
--Top 10 Aisles by Orders
SELECT a.aisle,
       COUNT(*) AS total_orders
FROM workspace.instacart.order_products op
JOIN workspace.instacart.products_clean p ON op.product_id = p.product_id
JOIN workspace.instacart.aisles a ON p.aisle_id = a.aisle_id
GROUP BY a.aisle
ORDER BY total_orders DESC
LIMIT 10;


aisle,total_orders
fresh fruits,3792661
fresh vegetables,3568630
packaged vegetables fruits,1843806
yogurt,1507583
packaged cheese,1021462
milk,923659
water seltzer sparkling water,878150
chips pretzels,753739
soy lactosefree,664493
bread,608469


In [0]:
--Cart Position vs Reorder Rate
SELECT add_to_cart_order,
       ROUND(AVG(reordered) * 100, 2) AS reorder_rate
FROM workspace.instacart.order_products
WHERE add_to_cart_order <= 10
GROUP BY add_to_cart_order
ORDER BY add_to_cart_order;


add_to_cart_order,reorder_rate
1,67.93
2,67.71
3,65.84
4,63.73
5,61.76
6,60.06
7,58.58
8,57.35
9,56.17
10,55.14


In [0]:
--Top 10 Reordered Products
SELECT p.product_name,
       COUNT(*) AS total_orders,
       SUM(op.reordered) AS reorders,
       ROUND(100.0 * SUM(op.reordered) / COUNT(*), 2)
           AS reorder_rate_pct
FROM workspace.instacart.order_products op
JOIN workspace.instacart.products_clean p ON op.product_id = p.product_id
GROUP BY p.product_name
HAVING total_orders > 500
ORDER BY reorder_rate_pct DESC
LIMIT 10;


product_name,total_orders,reorders,reorder_rate_pct
Half And Half Ultra Pasteurized,2995,2580,86.14
Whole Organic Omega 3 Milk,9410,8091,85.98
Organic Lactose Free Whole Milk,8742,7511,85.92
Organic Homogenized Whole Milk,4095,3514,85.81
Ultra-Purified Water,1524,1306,85.70
"Milk, Organic, Vitamin D",20770,17753,85.47
Organic Reduced Fat Milk,36869,31394,85.15
Goat Milk,5353,4551,85.02
Banana,491291,415166,84.51
Organic Lowfat 1% Milk,15352,12914,84.12


In [0]:
--Avg Basket Size per Department
SELECT d.department,
       ROUND(AVG(basket.items), 2) AS avg_items_per_order
FROM (
  SELECT op.order_id,
         p.department_id,
         COUNT(op.product_id) AS items
  FROM workspace.instacart.order_products op
  JOIN workspace.instacart.products_clean p ON op.product_id = p.product_id
  GROUP BY op.order_id, p.department_id
) basket
JOIN workspace.instacart.departments d
  ON basket.department_id = d.department_id
GROUP BY d.department
ORDER BY avg_items_per_order DESC;


department,avg_items_per_order
produce,3.95
dairy eggs,2.49
babies,2.38
snacks,2.08
frozen,1.9
beverages,1.85
alcohol,1.81
pantry,1.68
pets,1.65
household,1.57


In [0]:
--Top 5 Products per Department
WITH product_orders AS (
  SELECT d.department,
         p.product_name,
         COUNT(*) AS total_orders
  FROM workspace.instacart.order_products op
  JOIN workspace.instacart.products_clean p ON op.product_id = p.product_id
  JOIN workspace.instacart.departments d ON p.department_id = d.department_id
  GROUP BY d.department, p.product_name
),
ranked AS (
  SELECT *,
    RANK() OVER (
      PARTITION BY department
      ORDER BY total_orders DESC
    ) AS rnk
  FROM product_orders
)
SELECT department, product_name, total_orders, rnk
FROM ranked
WHERE rnk <= 5
ORDER BY department, rnk;


department,product_name,total_orders,rnk
alcohol,Sauvignon Blanc,8541,1
alcohol,Cabernet Sauvignon,6352,2
alcohol,Chardonnay,6346,3
alcohol,Beer,6068,4
alcohol,Vodka,5666,5
babies,Baby Food Stage 2 Blueberry Pear & Purple Carrot,9103,1
babies,Spinach Peas & Pear Stage 2 Baby Food,8303,2
babies,Gluten Free SpongeBob Spinach Littles,7342,3
babies,Broccoli & Apple Stage 2 Baby Food,7054,4
babies,Free & Clear Unscented Baby Wipes,6540,5


In [0]:
--Most Popular Product per Day of Week
WITH dow_product AS (
  SELECT o.order_dow,
         p.product_name,
         COUNT(*) AS total_orders
  FROM workspace.instacart.orders o
  JOIN workspace.instacart.order_products op ON o.order_id = op.order_id
  JOIN workspace.instacart.products_clean p ON op.product_id = p.product_id
  GROUP BY o.order_dow, p.product_name
),
ranked AS (
  SELECT *,
    ROW_NUMBER() OVER (
      PARTITION BY order_dow
      ORDER BY total_orders DESC
    ) AS rn
  FROM dow_product
)
SELECT order_dow, product_name, total_orders
FROM ranked
WHERE rn = 1
ORDER BY order_dow;


order_dow,product_name,total_orders
0,Banana,101474
1,Banana,90750
2,Banana,62002
3,Banana,55334
4,Banana,54447
5,Banana,61281
6,Banana,66003


In [0]:
--Customer RFM Table — Recency, Frequency, Monetary proxy
SELECT o.user_id,
       ROUND(AVG(o.days_since_prior_order), 1)
           AS recency_avg_days,
       COUNT(DISTINCT o.order_id) AS frequency,
       COUNT(op.product_id) AS monetary_items
FROM workspace.instacart.orders o
JOIN workspace.instacart.order_products op
  ON o.order_id = op.order_id
WHERE o.days_since_prior_order IS NOT NULL
GROUP BY o.user_id
ORDER BY frequency DESC
LIMIT 20;


user_id,recency_avg_days,frequency,monetary_items
8703,1.7,99,688
204485,2.7,99,692
99727,3.3,99,1284
40253,3.2,99,1567
171236,4.4,99,1613
152411,2.1,99,1263
154160,3.8,99,746
62890,3.5,99,902
68716,3.1,99,562
177893,3.2,99,1229


In [0]:
--Products High Volume, Low Reorder
WITH product_stats AS (
  SELECT p.product_name,
         COUNT(*) AS total_orders,
         ROUND(100.0 * SUM(op.reordered) / COUNT(*), 2)
             AS reorder_rate_pct
  FROM workspace.instacart.order_products op
  JOIN workspace.instacart.products_clean p ON op.product_id = p.product_id
  GROUP BY p.product_name
)
SELECT product_name, total_orders, reorder_rate_pct
FROM product_stats
WHERE total_orders > 1000
  AND reorder_rate_pct < 40
ORDER BY total_orders DESC
LIMIT 20;


product_name,total_orders,reorder_rate_pct
Organic Ketchup,15914,39.01
Light Brown Sugar,13853,27.60
Tomato Ketchup,13664,32.57
Original Hot Sauce,12543,39.52
Tomato Paste,11782,33.10
Organic Dijon Mustard,11503,25.16
Aluminum Foil,11181,21.88
Whole Milk Ricotta Cheese,10781,38.72
Organic Rosemary,10781,37.28
Chopped Walnuts,10471,36.31


In [0]:
--Top 3 First-Cart Products per Department
WITH first_adds AS (
  SELECT d.department,
         p.product_name,
         COUNT(*) AS first_add_count
  FROM workspace.instacart.order_products op
  JOIN workspace.instacart.products_clean p ON op.product_id = p.product_id
  JOIN workspace.instacart.departments d ON p.department_id = d.department_id
  WHERE op.add_to_cart_order = 1
  GROUP BY d.department, p.product_name
),
ranked AS (
  SELECT *,
    RANK() OVER (
      PARTITION BY department
      ORDER BY first_add_count DESC
    ) AS rnk
  FROM first_adds
)
SELECT department, product_name, first_add_count, rnk
FROM ranked
WHERE rnk <= 3
ORDER BY department, rnk;


department,product_name,first_add_count,rnk
alcohol,Vodka,2019,1
alcohol,Sauvignon Blanc,1905,2
alcohol,Beer,1702,3
babies,Organic Infant Formula With Iron,592,1
babies,Free & Clear Unscented Baby Wipes,565,2
babies,Gluten Free SpongeBob Spinach Littles,518,3
bakery,100% Whole Wheat Bread,6789,1
bakery,Organic Bread with 21 Whole Grains,2145,2
bakery,Ezekiel 4:9 Bread Organic Sprouted Whole Grain,1949,3
beverages,Spring Water,17552,1


In [0]:
--Rolling 3-Order Basket Size per Customer
WITH basket AS (
  SELECT o.user_id,
         o.order_number,
         COUNT(op.product_id) AS basket_size
  FROM workspace.instacart.orders o
  JOIN workspace.instacart.order_products op ON o.order_id = op.order_id
  GROUP BY o.user_id, o.order_number
)
SELECT user_id, order_number, basket_size,
  ROUND(AVG(basket_size) OVER (
    PARTITION BY user_id
    ORDER BY order_number
    ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
  ), 2) AS rolling_3_avg
FROM basket
WHERE user_id IN (
  SELECT user_id FROM workspace.instacart.orders
  GROUP BY user_id
  HAVING COUNT(DISTINCT order_id) >= 10
  LIMIT 5
)
ORDER BY user_id, order_number;

user_id,order_number,basket_size,rolling_3_avg
128,1,16,16.0
128,2,9,12.5
128,3,4,9.67
128,4,4,5.67
128,5,5,4.33
128,6,7,5.33
128,7,2,4.67
128,8,18,9.0
128,9,6,8.67
128,10,11,11.67


In [0]:
--Market Basket — Co-Purchased Products
SELECT p1.product_name AS product_1,
       p2.product_name AS product_2,
       COUNT(*) AS co_purchase_count
FROM workspace.instacart.order_products op1
JOIN workspace.instacart.order_products op2
  ON op1.order_id = op2.order_id
  AND op1.product_id < op2.product_id
JOIN workspace.instacart.products_clean p1 ON op1.product_id = p1.product_id
JOIN workspace.instacart.products_clean p2 ON op2.product_id = p2.product_id
GROUP BY p1.product_name, p2.product_name
HAVING co_purchase_count >= 1000
ORDER BY co_purchase_count DESC
LIMIT 15;


product_1,product_2,co_purchase_count
Bag of Organic Bananas,Organic Hass Avocado,64761
Bag of Organic Bananas,Organic Strawberries,64702
Organic Strawberries,Banana,58330
Banana,Organic Avocado,55611
Organic Baby Spinach,Banana,53395
Bag of Organic Bananas,Organic Baby Spinach,52608
Strawberries,Banana,43180
Banana,Large Lemon,43038
Organic Strawberries,Organic Hass Avocado,42333
Bag of Organic Bananas,Organic Raspberries,42283


In [0]:
--— Loyal Customer vs One-Time Product Preference
WITH user_seg AS (
  SELECT user_id,
    CASE
      WHEN COUNT(DISTINCT order_id) >= 16 THEN 'loyal'
      WHEN COUNT(DISTINCT order_id) = 1   THEN 'onetime'
      ELSE 'other'
    END AS segment
  FROM workspace.instacart.orders
  GROUP BY user_id
),
dept_orders AS (
  SELECT d.department,
         us.segment,
         COUNT(*) AS orders
  FROM workspace.instacart.orders o
  JOIN user_seg us ON o.user_id = us.user_id
  JOIN workspace.instacart.order_products op ON o.order_id = op.order_id
  JOIN workspace.instacart.products_clean p ON op.product_id = p.product_id
  JOIN workspace.instacart.departments d ON p.department_id = d.department_id
  WHERE us.segment IN ('loyal','onetime')
  GROUP BY d.department, us.segment
)
SELECT department,
  SUM(CASE WHEN segment='loyal'   THEN orders ELSE 0 END) AS loyal_orders,
  SUM(CASE WHEN segment='onetime' THEN orders ELSE 0 END) AS onetime_orders
FROM dept_orders
GROUP BY department
ORDER BY loyal_orders DESC;


department,loyal_orders,onetime_orders
produce,7052905,0
dairy eggs,4044668,0
snacks,2135765,0
beverages,1968963,0
frozen,1557822,0
pantry,1329030,0
bakery,863273,0
deli,761288,0
canned goods,740269,0
dry goods pasta,603601,0


In [0]:
-- Products in Prior but Not Train
SELECT p.product_id, p.product_name
FROM workspace.instacart.products_clean p
WHERE p.product_id IN (
  SELECT product_id FROM workspace.instacart.order_products_prior
  EXCEPT
  SELECT product_id FROM workspace.instacart.order_products_train
)
ORDER BY p.product_name
LIMIT 20;


product_id,product_name
34207,"""""""Constant Comment\"""" Green Tea Blend Tea Bags"""
44344,"""""""Louis Ba-Kahn\"""" Chocolate Chip Cookie & Brown Butter Candied Bacon Ice Cream Sandwich"""
29882,""".5"""" Waterproof Tape"""
23454,"""10"""" Square Summer Plates"""
36748,"""10.25"""" Cast Iron Skillet"""
15447,"""11"""" Salad Tongs"""
5066,"""2"""" x 6\"""" Unscented White Pillar Candle"""
23167,"""3"""" Palm Wax Pillar Candle"""
28152,"""3.5"""" Aluminum Pan"""
37976,"""5"""" Santoku Knife"""


In [0]:
--Common Products in Both Sets
WITH common AS (
  SELECT product_id FROM workspace.instacart.order_products_prior
  INTERSECT
  SELECT product_id FROM workspace.instacart.order_products_train
),
prior_cnt AS (
  SELECT product_id, COUNT(*) AS prior_orders
  FROM workspace.instacart.order_products_prior
  WHERE product_id IN (SELECT product_id FROM common)
  GROUP BY product_id
),
train_cnt AS (
  SELECT product_id, COUNT(*) AS train_orders
  FROM workspace.instacart.order_products_train
  WHERE product_id IN (SELECT product_id FROM common)
  GROUP BY product_id
)
SELECT p.product_name, pc.prior_orders, tc.train_orders
FROM prior_cnt pc
JOIN train_cnt tc ON pc.product_id = tc.product_id
JOIN workspace.instacart.products_clean p ON pc.product_id = p.product_id
ORDER BY prior_orders DESC
LIMIT 10;


product_name,prior_orders,train_orders
Banana,472565,18726
Bag of Organic Bananas,379450,15480
Organic Strawberries,264683,10894
Organic Baby Spinach,241921,9784
Organic Hass Avocado,213584,7293
Organic Avocado,176815,7409
Large Lemon,152657,8135
Strawberries,142951,6494
Limes,140627,6033
Organic Whole Milk,137905,4908
